In [ ]:
%pip install qiskit
%pip install qiskit_aer
%pip install qiskit_ibm_runtime
%pip install pylatexenc

In [2]:
import qiskit
from qiskit import QuantumCircuit, transpile, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.visualization import plot_circuit_layout, circuit_drawer, plot_circuit_layout


from math import pi

#sim_aer = AerSimulator()
fake_quebec = FakeSherbrooke()
pm = generate_preset_pass_manager(optimization_level = 3, backend = fake_quebec)

In [8]:
def qft_circuit(n_qubits: int) -> QuantumCircuit:
  q = QuantumRegister(n_qubits+2)
  c = ClassicalRegister(1)

  qc = QuantumCircuit(q,c)
  qc.h(q[0])
  qc.cp(pi/2, q[0], q[1])
  qc.h(q[2])
  qc.cx(q[2],q[3])
  qc.cx(q[0],q[2])
  qc.measure(q[2],c)
  with qc.if_test((c,1)):
    qc.x(q[2])
    qc.x(q[3])

  qc.cp(pi/4,3,4)
  qc.cp(pi/8,3,5)
  qc.h(3)
  qc.measure(q[3],c)
  with qc.if_test((c,1)):
    qc.z(q[0])
    qc.x(q[3])

  qc.barrier()

  qc.h(q[1])
  qc.h(q[2])
  qc.cx(q[2],q[3])
  qc.cx(q[1],q[2])
  qc.measure(q[2],c)
  with qc.if_test((c,1)):
    qc.x(q[2])
    qc.x(q[3])
  qc.barrier()

  qc.cp(pi/2,3,4)
  qc.cp(pi/4,3,5)
  qc.h(3)
  qc.measure(3,c)
  with qc.if_test((c,1)):
    qc.z(q[1])
    qc.x(q[3])

  qc.barrier()

  qc.h(4)
  qc.cp(pi/2,5,4)
  qc.h(5)

  return qc

qc = qft_circuit(4)
qc.draw('mpl')

In [ ]:
#q1 -> ctrl; q2 -> com1; q3 -> com2;
def cat_entangler() -> QuantumCircuit:
  cat_qc = QuantumCircuit(3,1)
  cat_qc.h(1)
  cat_qc.cx(1,2)

  cat_qc.cx(0,1)
  cat_qc.measure(1,0)
  with cat_qc.if_test((0,1)):
    cat_qc.x(1)
    cat_qc.x(2)

  return cat_qc

thingy = cat_entangler()
thingy.draw('mpl')


In [ ]:
#q1 -> ctrl; q2 -> com2;
def cat_disentangler() -> QuantumCircuit:
  uncat_qc = QuantumCircuit(2,1)
  uncat_qc.h(1)
  uncat_qc.measure(1,0)
  with uncat_qc.if_test((0,1)):
    uncat_qc.z(0)
    uncat_qc.x(1)

  return uncat_qc

the_thingy = cat_disentangler()
the_thingy.draw('mpl')

In [64]:

#assumes one communicative qubit per device with shared communication
def qft_circuit_dist(n_qubits: int, n_devices : int, device_size : int) -> QuantumCircuit:
  if n_qubits > n_devices*device_size:
    print('not enough device space')
    return

  c = ClassicalRegister(1, 'cl_comm')
  qc = QuantumCircuit(c)


  #qubit allocation
  full_devices = min(n_qubits // device_size, n_devices)
  remainder = n_qubits % device_size

  for i in range(full_devices):
    qc.add_register(QuantumRegister(device_size, f'dev{i}'))

  if full_devices < n_devices and remainder > 0:
    qc.add_register(QuantumRegister(remainder, f'dev{full_devices}'))

  qc.add_register(QuantumRegister(n_devices, 'q_comm'))



  #circuit logic

  #loops through devices
  for dev_idx, qreg in enumerate(qc.qregs[:-1]):

    print(f"QuantumRegister: {qreg.name}")

    for i, idx_qubit in enumerate(qreg):
      qc.barrier()
      angleCoeff = 1
      print(f"  Current qubit: {idx_qubit}")
      qc.h(idx_qubit)
      for next_qubit in qreg[i+1:]:
        print(f"    Later qubit in same device: {next_qubit}")
        qc.cp(pi/(2**angleCoeff), idx_qubit, next_qubit)
        angleCoeff += 1

      qc.barrier()

      #looping subsequent devices
      for later_dev_idx, later_dev in enumerate(qc.qregs[dev_idx + 1:-1], dev_idx+1):
        #cat entangler
        qc.barrier()
        qc.h(qc.qregs[-1][dev_idx])
        qc.cx(qc.qregs[-1][dev_idx], qc.qregs[-1][later_dev_idx])

        qc.cx(idx_qubit, qc.qregs[-1][dev_idx])
        qc.measure(qc.qregs[-1][dev_idx], c)
        with qc.if_test((c,1)):
          qc.x(qc.qregs[-1][dev_idx])
          qc.x(qc.qregs[-1][later_dev_idx])

        qc.barrier()

        for other_qubit in later_dev:
          print(f"    Qubit from subsequent device {later_dev.name}: {other_qubit}")
          qc.cp(pi/(2**angleCoeff), qc.qregs[-1][later_dev_idx], other_qubit)
          angleCoeff += 1


        #cat disentangler
        qc.barrier()
        qc.h(qc.qregs[-1][later_dev_idx])
        qc.measure(qc.qregs[-1][later_dev_idx], c)
        with qc.if_test((c,1)):
          qc.z(idx_qubit)
          qc.x(qc.qregs[-1][later_dev_idx])
        qc.barrier()



  return qc



dist_qft = qft_circuit_dist(4,2,2)
dist_qft.draw('mpl')

QuantumRegister: dev0
  Current qubit: <Qubit register=(2, "dev0"), index=0>
    Later qubit in same device: <Qubit register=(2, "dev0"), index=1>
    Qubit from subsequent device dev1: <Qubit register=(2, "dev1"), index=0>
    Qubit from subsequent device dev1: <Qubit register=(2, "dev1"), index=1>
  Current qubit: <Qubit register=(2, "dev0"), index=1>
    Qubit from subsequent device dev1: <Qubit register=(2, "dev1"), index=0>
    Qubit from subsequent device dev1: <Qubit register=(2, "dev1"), index=1>
QuantumRegister: dev1
  Current qubit: <Qubit register=(2, "dev1"), index=0>
    Later qubit in same device: <Qubit register=(2, "dev1"), index=1>
  Current qubit: <Qubit register=(2, "dev1"), index=1>


In [ ]:
tdist_qft = pm.run(dist_qft)
tdist_qft.draw('mpl')

In [ ]:
plot_circuit_layout(tdist_qft, fake_quebec, view="physical")